In [3]:
import shutil
import pandas as pd
import random

import copy
import numpy as np # Import numpy for checking finite values
import matplotlib.pyplot as plt # Import matplotlib for potential debugging

import os
import math # Import math for ceil
from sklearn.manifold import TSNE # Import TSNE to check default perplexity



import os
import glob
import json
import numpy as np
import tensorflow as tf
import sys

In [4]:

fixed_path = 'C:\\Users\\admin\\0.Master_Thesis\\'
#utils_path = 'C:\\Users\\Enrico Didoli\\0.Master_Thesis\\'

if fixed_path not in sys.path:
    sys.path.append(fixed_path)
    
cellcnn_path = f'{fixed_path}CellCNN'
if cellcnn_path not in sys.path:
    sys.path.append(cellcnn_path)

general_functions_path = f'{fixed_path}General_Functions/'
if general_functions_path not in sys.path:
    sys.path.append(general_functions_path)



#cell_repo = '/content/drive/MyDrive//Master Thesis/Code_trial_1/B-ALL_expression_matrix.csv'


In [5]:
decache_files = ['timepoints_elaboration', 'results_elaboration', 'new_datasets_generation',
                 'cellCnn.model_new_unfixed', 'cellCnn.utils_new_unfixed', 
                 'cellCnn.run_models', 'cellCnn.downsample_new_unfixed']

# Rimuovi il modulo specifico dalla cache
from timepoints_elaboration import remove_from_cache
remove_from_cache(decache_files)

# import downloaded modules
from cellCnn.model_new_unfixed import CellCnn
import cellCnn.utils_new_unfixed as utils
import cellCnn.downsample_new_unfixed as downsample

from timepoints_elaboration import load_data, donation_extraction, dataset_elaboration
from timepoints_elaboration import donor_division, patient_code_extraction, remove_from_cache

from results_elaboration import extract_hyper, phenotype_prediction, default_serializer, show_hyperparameters
from results_elaboration import elaborate_predictions, show_hyper

from run_models import trials_train_CellCNN, trials_test_CellCNN

from new_datasets_generation import splitting_and_dataset_elaboration


timepoints_elaboration rimosso dalla cache
results_elaboration non trovato nella cache
new_datasets_generation non trovato nella cache
cellCnn.model_new_unfixed non trovato nella cache
cellCnn.utils_new_unfixed non trovato nella cache
cellCnn.run_models non trovato nella cache
cellCnn.downsample_new_unfixed non trovato nella cache


# Import datasets from .csv

In [6]:
%%time

# Trova tutti i file con estensione specifica in una cartella
data_folder = f'{fixed_path}B-ALL_Datasets'
extension = '*.csv'  # cambia con l'estensione che ti serve

multiple_donations, ALL_DATASETS = load_data(data_path = data_folder, ext = extension)

Elaborating file 0: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220204-2900.csv
Elaborating file 1: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220208-3665.csv
Elaborating file 2: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220216-3546.csv
Elaborating file 3: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D15_1.csv
Elaborating file 4: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D29_1.csv
Elaborating file 5: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D71_1.csv
Elaborating file 6: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE12_D15_2.csv
Elaborating file 7: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE12_D29_1.csv
Elaborating file 8: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE1_D15_2.csv
Elaborating

In [7]:
for i, dataset in enumerate(ALL_DATASETS):
    blast_n = (dataset['IsBlast'] == 1).sum()
    
    print(f'dataset {i} has {blast_n} cells over {len(dataset)} cells')

dataset 0 has 0 cells over 6558750 cells
dataset 1 has 0 cells over 2764877 cells
dataset 2 has 0 cells over 1291729 cells
dataset 3 has 1348 cells over 843500 cells
dataset 4 has 170 cells over 1404000 cells
dataset 5 has 0 cells over 3265250 cells
dataset 6 has 639 cells over 430869 cells
dataset 7 has 15 cells over 722000 cells
dataset 8 has 757 cells over 864000 cells
dataset 9 has 0 cells over 1947518 cells
dataset 10 has 308059 cells over 778750 cells
dataset 11 has 830101 cells over 5510750 cells
dataset 12 has 72 cells over 208000 cells
dataset 13 has 0 cells over 2912500 cells
dataset 14 has 3096 cells over 160500 cells
dataset 15 has 1449 cells over 3591480 cells
dataset 16 has 0 cells over 3107000 cells
dataset 17 has 9147 cells over 637409 cells
dataset 18 has 227 cells over 2928000 cells
dataset 19 has 518 cells over 164553 cells
dataset 20 has 398 cells over 191975 cells
dataset 21 has 0 cells over 169000 cells
dataset 22 has 2390 cells over 479000 cells
dataset 23 has 77

In [8]:
# Show patients donations
print(multiple_donations)
#['12', '9', '13', '7', '11'] 

{'11': [3, 4, 5], '12': [6, 7], '1': [8, 9], '2': [10, 11], '3': [12, 13], '4': [14, 15, 16], '6': [17, 18], '7': [19, 20, 21], '8': [22, 23], '9': [24, 25], '13': [0], '14': [1], '15': [2]}


In [9]:
healthy_donors, blast_donors, mixed_donors = donor_division(multiple_donations, ALL_DATASETS)

print(healthy_donors, blast_donors, mixed_donors)

{'11': [1, 1, 0], '12': [1, 1], '1': [1, 0], '2': [1, 1], '3': [1, 0], '4': [1, 1, 0], '6': [1, 1], '7': [1, 1, 0], '8': [1, 1], '9': [1, 1], '13': [0], '14': [0], '15': [0]}
['12', '2', '6', '8', '9'] ['13', '14', '15'] ['11', '1', '3', '4', '7']


In [10]:

# Samples donors for Train, Validation and Test sets    
train_donors_idx, val_donors_idx, test_donors_idx = dataset_elaboration(multiple_donations, ALL_DATASETS, healthy_donors, blast_donors,
                        mixed_donors, seed = 42)



Precess starts. Dividing donors...
healthy_donors_idx, blast_donors_idx, mixed_donors_idx: [0, 4, 2, 1, 3], [0, 2, 1],[4, 0, 2, 1, 3]
Seting Train, Validation and Test idx...
['12', '9', '13', '7'] ['6', '15', '3'] ['2', '8', '14', '11', '1', '4']


In [11]:

#  Retrieves specific donor datasets from all datasets list
train_datasets_extracted = donation_extraction(train_donors_idx, multiple_donations, ALL_DATASETS)
val_datasets_extracted = donation_extraction(val_donors_idx, multiple_donations, ALL_DATASETS)
test_datasets_extracted = donation_extraction(test_donors_idx, multiple_donations, ALL_DATASETS)


print(len(train_datasets_extracted)) # list of donators
print(len(train_datasets_extracted[0])) # list of donations
print(len(test_datasets_extracted[0][0])) # dataset of cells

4
2
778750


In [12]:
n_sub = 3
seed = 42
n_cells = 100000
new_train_datasets, new_train_y, new_val_datasets, new_val_y, new_test_datasets, new_test_y = splitting_and_dataset_elaboration(train_datasets_extracted, 
                                                                                    val_datasets_extracted, test_datasets_extracted,
                                                                                    n_sub, n_cells, seed)



New training datasets creation...
4
2

New Donor
2
Tot blast data in the donor timepoints: 639
Tot blast data in the donor timepoints: 654
Extraction Done
Condition: 1
Chosen # of blast cells: [10000. 20000.  5000.]
Chosen # of blast cells: [ 5000. 10000. 20000.]

New Donor
2
Tot blast data in the donor timepoints: 44697
Tot blast data in the donor timepoints: 46110
Extraction Done
Condition: 1
Chosen # of blast cells: [20000.   500.  1000.]
Chosen # of blast cells: [  500.  1000. 20000.]

New Donor
1
Timepoint condition: Healthy
Extraction Done
Condition: 0

New Donor
3
Tot blast data in the donor timepoints: 518
Tot blast data in the donor timepoints: 916
Tot blast data in the donor timepoints: 916
Extraction Done
Condition: 1
Chosen # of blast cells: [10000. 10000. 20000.]
Chosen # of blast cells: [10000. 10000. 20000.]
[1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0]
Done!

New training datasets creation...

New Donor
Tot blast data in the donor timepoints: 9147
Tot 

In [13]:
for i, dataset in enumerate(new_test_datasets):
    blast_n = (dataset['IsBlast'] == 1).sum()
    
    print(f'dataset {i} has {blast_n} cells over {len(dataset)} cells. Label: {new_test_y[i]}')

dataset 0 has 500 cells over 100000 cells. Label: 1
dataset 1 has 5000 cells over 100000 cells. Label: 1
dataset 2 has 20000 cells over 100000 cells. Label: 1
dataset 3 has 0 cells over 100000 cells. Label: 0
dataset 4 has 0 cells over 100000 cells. Label: 0
dataset 5 has 0 cells over 100000 cells. Label: 0
dataset 6 has 500 cells over 100000 cells. Label: 1
dataset 7 has 500 cells over 100000 cells. Label: 1
dataset 8 has 10000 cells over 100000 cells. Label: 1
dataset 9 has 0 cells over 100000 cells. Label: 0
dataset 10 has 0 cells over 100000 cells. Label: 0
dataset 11 has 0 cells over 100000 cells. Label: 0
dataset 12 has 0 cells over 100000 cells. Label: 0
dataset 13 has 0 cells over 100000 cells. Label: 0
dataset 14 has 0 cells over 100000 cells. Label: 0
dataset 15 has 10000 cells over 100000 cells. Label: 1
dataset 16 has 20000 cells over 100000 cells. Label: 1
dataset 17 has 20000 cells over 100000 cells. Label: 1
dataset 18 has 0 cells over 100000 cells. Label: 0
dataset 19 h

In [14]:
print(len(new_train_datasets))
print(len(new_train_y))
print(len(new_val_datasets))
print(len(new_val_y))
print(len(new_test_datasets))
print(len(new_test_y))


21
21
15
15
33
33


In [15]:
original_test_datasets = []
original_test_y = []
for donor in test_datasets_extracted:
    for dataset in donor:
        
        if len(dataset[dataset['IsBlast'] == 1]) > 0:
            original_test_y.append(1)
            
        else:
            original_test_y.append(0)
        dataset = dataset.drop(columns = ['IsBlast'])
        original_test_datasets.append(dataset)
print(len(original_test_datasets))
print(original_test_y)

13
[1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0]


In [16]:
new_no_label_test_datasets = []
for dataset in new_test_datasets:
    dataset = dataset.drop(columns = ['IsBlast'])
    new_no_label_test_datasets.append(dataset)
print(len(new_no_label_test_datasets))

33


In [17]:
decache_files = ['timepoints_elaboration', 'results_elaboration', 'new_datasets_generation',
                 'cellCnn.model_new_unfixed', 'cellCnn.utils_new_unfixed', 
                 'cellCnn.run_models', 'cellCnn.downsample_new_unfixed']

# Rimuovi il modulo specifico dalla cache
from timepoints_elaboration import remove_from_cache
remove_from_cache(decache_files)

# import downloaded modules
from cellCnn.model_new_unfixed import CellCnn
import cellCnn.utils_new_unfixed as utils
import cellCnn.downsample_new_unfixed as downsample

from timepoints_elaboration import load_data, donation_extraction, dataset_elaboration
from timepoints_elaboration import donor_division, patient_code_extraction, remove_from_cache

from results_elaboration import extract_hyper, phenotype_prediction, default_serializer, show_hyperparameters
from results_elaboration import elaborate_predictions, show_hyper

from run_models import trials_train_CellCNN, trials_test_CellCNN

from new_datasets_generation import splitting_and_dataset_elaboration


timepoints_elaboration rimosso dalla cache
results_elaboration rimosso dalla cache
new_datasets_generation rimosso dalla cache
cellCnn.model_new_unfixed rimosso dalla cache
cellCnn.utils_new_unfixed rimosso dalla cache
cellCnn.run_models non trovato nella cache
cellCnn.downsample_new_unfixed rimosso dalla cache


In [20]:
%%time
model_setting = 4
seed = 42
seed_list = [7359, 9654, 4557, 106, 2615, 6924, 5574, 4552, 2547, 3527]

models_list = trials_train_CellCNN(CellCnn, new_train_datasets, new_train_y,
                    new_val_datasets, new_val_y, 
                    new_no_label_test_datasets, 
                    trials = 2,
                    max_epochs = 1, nrun = 1, n_cell = n_cells, seed_list = seed_list)

Trial 1 started
Seed used: 7359
Model defined...
Fitting started...

# ============================================= #
Run: 0

Seed: 7359
1/1 [==============================] - 0s 196ms/step - loss: 0.5733

# ============================================= #
Run: 1

Seed: 7360
1/1 [==============================] - 0s 142ms/step - loss: 0.6892

# ============================================= #
Run: 2

Seed: 7361
1/1 [==============================] - 0s 281ms/step - loss: 0.6374
Trial 2 started
Seed used: 19308
Model defined...
Fitting started...

# ============================================= #
Run: 0

Seed: 19308
1/1 [==============================] - 0s 495ms/step - loss: 0.6895

# ============================================= #
Run: 1

Seed: 19309
1/1 [==============================] - 0s 267ms/step - loss: 0.6834

# ============================================= #
Run: 2

Seed: 19310
1/1 [==============================] - 0s 480ms/step - loss: 1.1145
CPU times: total: 55.9 s
Wall ti

In [21]:
print(len(models_list))
predictions_list, results_list = trials_test_CellCNN(models_list, new_no_label_test_datasets)
#print(len(predictions_list))

2
2
Prediction started...


C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib

2/2 [==============================] - 0s 19ms/step
Done
Trial 1 Done!

2
Prediction started...


C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\admin\anaconda3\envs\tf_env\lib

2/2 [==============================] - 0s 23ms/step
Done
Trial 2 Done!



In [ ]:
save_path = f'{fixed_path}\\results\\'
with open(f'{save_path}14_10_new_results_list_{model_setting}.json', "w", encoding="utf-8") as f:
    json.dump(tot_trials_res, f, default=default_serializer, ensure_ascii=False, indent=2)

with open(f'{save_path}14_10_new_predictions_list_{model_setting}.json', "w", encoding="utf-8") as f:
    json.dump(predictions_list, f, default=default_serializer, ensure_ascii=False, indent=2)

with open(f'{save_path}14_10_new_test_y_{model_setting}.json', "w", encoding="utf-8") as f:
    json.dump(new_test_y, f, default=default_serializer, ensure_ascii=False, indent=2)

In [22]:
tot_trials_res = []
par_list = ['config', 'model_sorted_idx']

for res in results_list:
    needed_results = {}
    for key, value in res.items():
        if key in par_list:
            print(key, value)
            needed_results[key] = value
    tot_trials_res.append(needed_results)


config {'nfilter': [5, 9, 5], 'learning_rate': [0.1, 0.001, 0.1], 'maxpool_percentage': [5.0, 0.01, 100.0]}
model_sorted_idx [1 2 0]
config {'nfilter': [9, 3, 9], 'learning_rate': [0.001, 0.1, 0.1], 'maxpool_percentage': [20.0, 100.0, 20.0]}
model_sorted_idx [2 0 1]


In [23]:
new_best_per_trial = show_hyper(results_list, best_3 = False)
print(new_best_per_trial)


   nfilter  learning_rate  maxpool_percentage
0      9.0          0.001                0.01
1      9.0          0.100               20.00


In [ ]:
pred_phenotype_df, accuracy_list = elaborate_predictions(predictions_list, new_test_y, results = True)


In [ ]:

original_predictions_list, original_results_list = test_CellCNN_trials(models_list, original_test_datasets)
#print(len(predictions_list))

In [ ]:
save_path = f'{fixed_path}\\results\\'
with open(f'{save_path}14_10_original_results_list_{model_setting}.json', "w", encoding="utf-8") as f:
    json.dump(original_tot_trials_res, f, default=default_serializer, ensure_ascii=False, indent=2)

with open(f'{save_path}14_10_original_predictions_list_{model_setting}.json', "w", encoding="utf-8") as f:
    json.dump(original_predictions_list, f, default=default_serializer, ensure_ascii=False, indent=2)

with open(f'{save_path}14_10_original_test_y_{model_setting}.json', "w", encoding="utf-8") as f:
    json.dump(original_test_y, f, default=default_serializer, ensure_ascii=False, indent=2)

In [ ]:
original_best_per_trial = show_hyper(original_results_list, best_3 = False)
print(original_best_per_trial)

In [ ]:
original_pred_phenotype_df, original_accuracy_list = elaborate_predictions(original_predictions_list, original_test_y, results = True)

In [ ]:
original_tot_trials_res = []
par_list = ['config', 'model_sorted_idx']

for res in original_results_list:
    needed_results = {}
    for key, value in res.items():
        if key in par_list:
            print(key, value)
            needed_results[key] = value
    original_tot_trials_res.append(needed_results)
#print(tot_trials_res)